# mini-AGI v16.2 — Empirical Validation Campaign (Phases 1-9)

Runs the gated validation pipeline against the released v16.1 ZIP on a clean
GPU runtime. Each gate must pass before the next phase starts.

**Gate rule:** do not train anything until release + environment validation
passes. Do not proceed to Campaign 1 until the Qwen smoke produces evidence.

Inputs:
- `mini-AGI-v16.1-Colab-Converged-Full-Upgraded.zip` uploaded to `/content` (or Drive)
- `v162-validation-pack.zip` uploaded to `/content` (contains `scripts/validation/`)

Outputs land in `/content/drive/MyDrive/mini-AGI-v16.2-validation/`.

In [ ]:
RELEASE = 'mini-AGI-v16.1-Colab-Converged-Full-Upgraded'
ROOT = f'/content/{RELEASE}'
WORK = '/content/drive/MyDrive/mini-AGI-v16.2-validation'
VALPACK = '/content/v162-validation-pack'
VAL = f'{VALPACK}/scripts/validation'
import os
os.makedirs(WORK, exist_ok=True)

## Phase 1 — Clean-environment release verification
Fresh runtime (GPU enabled), Drive mount, unzip, verify all 1,381 manifest
entries + signature, run the Colab doctor, save `READINESS_REPORT.json`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Upload both zips to /content first (left file pane), or copy from Drive:
# !cp '/content/drive/MyDrive/mini-AGI-v16.1-Colab-Converged-Full-Upgraded.zip' /content/
# !cp '/content/drive/MyDrive/v162-validation-pack.zip' /content/
!unzip -q -o /content/mini-AGI-v16.1-Colab-Converged-Full-Upgraded.zip -d /content/
!unzip -q -o /content/v162-validation-pack.zip -d /content/v162-validation-pack
%cd /content/mini-AGI-v16.1-Colab-Converged-Full-Upgraded

In [ ]:
!bash scripts/colab_install.sh
!python -m pip install -q pytest

In [ ]:
import json, subprocess, sys

def run(cmd, **kw):
    print('+', ' '.join(cmd), flush=True)
    r = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True, **kw)
    sys.stdout.write(r.stdout); sys.stderr.write(r.stderr)
    return r

# GATE 1a: release integrity — all 1,381 manifest entries + signature
r = run(['python', 'scripts/verify_release.py'])
assert r.returncode == 0 and json.loads(r.stdout)['files_verified'] == 1381, 'RELEASE VERIFICATION FAILED'

# GATE 1b: environment — GPU required
r = run(['python', '-m', 'minagi.platforms.colab.doctor', '--output', '/content/READINESS_REPORT.json'])
report = json.loads(open('/content/READINESS_REPORT.json').read())
assert r.returncode == 0 and report['ready_for_gpu_campaign'], f'ENVIRONMENT NOT READY: {report["reasons"]}'
!cp /content/READINESS_REPORT.json {WORK}/READINESS_REPORT.json
print('GATE 1 PASSED:', report['environment']['gpu_name'], report['environment']['gpu_vram_bytes']//2**20, 'MiB VRAM')

## Phase 2 — Reproduce software validation (313/313 + imports)

In [ ]:
# GATE 2a: unified test suite must reproduce 313/313 (documented exclusions only)
r = run(['python', '-m', 'pytest', 'tests-python', '-q'])
assert r.returncode == 0, 'TEST SUITE FAILED IN COLAB'
tail = r.stdout.strip().splitlines()[-1]
assert '313 passed' in tail, f'unexpected test count: {tail}'
print('GATE 2a PASSED:', tail)

# GATE 2b: import probe
r = run(['python', f'{VAL}/import_probe.py', '--root', ROOT, '--output', f'{WORK}/IMPORT_PROBE.json'])
assert r.returncode == 0, 'IMPORT PROBE FAILED'
print('GATE 2 PASSED')

## Phase 3 — Tiny real-neural LoRA smoke test
Frozen tiny-gpt2 → A0 → frozen dataset → real LoRA gradient updates →
`adapter_model.safetensors` → SHA-256 binding → destroy model → reload → A1.

In [ ]:
r = run(['python', 'scripts/run_colab_campaign.py', '--config', 'configs/smoke.yaml', '--storage', WORK])
assert r.returncode == 0, 'TINY SMOKE CAMPAIGN CRASHED'
res = json.loads(open(f'{WORK}/campaigns/colab-smoke-v161/RESULT.json').read())
assert res['results'], 'no seed results'
print('PHASE 3 COMPLETE — decision:', res['decision'], '| delta:', res['mean_forward_transfer_delta'])

## Phases 4-6 — Adversarial boundary checks
Every corruption/substitution must be **REJECT**ed.

In [ ]:
r = run(['python', f'{VAL}/adversarial_checks.py', '--root', ROOT, '--output', f'{WORK}/ADVERSARIAL_CHECKS.json'])
assert r.returncode == 0, 'AN ADVERSARIAL CHECK FAILED TO REJECT'
print('PHASES 4-6 PASSED — 16 rejection checks')

## Phase 7 — Interruption recovery
1. Rule-level selftest (fabricated completed/incomplete/missing states).
2. Live check: delete `seed-0/COMPLETE` from the smoke campaign, re-run the
   campaign, verify the cell is resumed (not trusted as stale evidence) and
   completed cells are not silently re-executed.
3. Manual: after starting a campaign, use *Runtime → Disconnect and delete
   runtime*, reconnect, re-run the same cell — Drive-persisted runs resume.

In [ ]:
r = run(['python', f'{VAL}/interruption_recovery.py', '--root', ROOT, '--selftest',
         '--campaign-dir', f'{WORK}/campaigns/colab-smoke-v161',
         '--output', f'{WORK}/INTERRUPTION_RECOVERY.json'])
assert r.returncode == 0
# audit the real smoke campaign dir
rep = json.loads(open(f'{WORK}/INTERRUPTION_RECOVERY.json').read())
print(rep['classification'])
print('PHASE 7 PASSED')

## Phase 8 — Qwen LoRA smoke campaign
A0 frozen Qwen2.5-0.5B vs A1 same + LoRA; everything else fixed.

In [ ]:
r = run(['python', 'scripts/run_colab_campaign.py', '--config', 'configs/qwen_smoke.yaml', '--storage', WORK])
assert r.returncode == 0, 'QWEN SMOKE CRASHED'
res = json.loads(open(f'{WORK}/campaigns/qwen-smoke-v161/RESULT.json').read())
print('PHASE 8 COMPLETE — decision:', res['decision'], '| delta:', res['mean_forward_transfer_delta'])

## Phase 9 — Prove the qualified adapter genuinely affects Qwen
Fresh A0 → adapter → A1 (must differ) → adapter removed → A0 restored.
Also verifies the on-disk adapter digest equals the receipt-bound digest.

In [ ]:
r = run(['python', f'{VAL}/adapter_effect_check.py', '--root', ROOT,
         '--config', 'configs/qwen_smoke.yaml', '--storage', WORK, '--seed', '0',
         '--output', f'{WORK}/ADAPTER_EFFECT_REPORT.json'])
assert r.returncode == 0, 'ADAPTER EFFECT CHECK FAILED'
print('PHASE 9 PASSED — adapter provably alters and un-alters inference')

## Phases 16-17 (smoke scope) — Independent artifact-only qualification
The qualifier reads only persisted artifacts, verifies signatures + digests,
refuses incomplete matrices, and re-derives the decision independently.

In [ ]:
for cid in ('colab-smoke-v161', 'qwen-smoke-v161'):
    r = run(['python', f'{VAL}/qualify_campaign.py', '--root', ROOT,
             '--storage', WORK, '--campaign-id', cid])
    rep = json.loads(open(f'{WORK}/campaigns/{cid}/QUALIFICATION_RECORD.json').read())
    print(cid, '->', rep['decision'], '| runner agreement:', rep['runner_decision_agreement'])
print('Validation bundle complete ->', WORK)

## STOP — Phases 10+ are gated

Do not proceed to the six-arm experiment or Campaign 1 until the above
produces evidence on this runtime. Remaining v16.2 work, in order:

- **Phase 10** six-arm runner (Frozen / Retrieval / Semantic Memory / Skills /
  Grounded Replay / Neural Adapter) — requires arm-schema extension beyond
  `{A0,A1}` and wiring existing L2-L5 mechanisms to the HF evaluation path.
- **Phase 11-13** acquisition/retention/FT/security/efficiency split,
  negative controls, N=5 preregistered seeds with full stats.
- **Phase 14** sign + freeze `CAMPAIGN_1_PLAN.json` (pin immutable HF commit,
  real preregistered dataset — `campaign1.yaml` is a placeholder by design).
- **Phase 15-19** witnessed run matrix, independent qualification,
  fresh-runtime reproduction (`REPRODUCTION_RECORD.json`), canary epoch.
- **Phase 20** assemble v16.2 as an *evidence* release.